---

# Sprint 5 - fusion patch, closing the week 2 task

The fusion run concluded that no method beat the stronger single view, and froze `cc_only`.
That is correct as a *finding* and wrong as an *operational rule*: 3,364 breasts in the
development set carry only MLO, and `cc_only` cannot score them.

This patch separates the two. The finding stays as it is - fusion was not supported. The
operational rule becomes the maximum over whichever views exist, which is what
`risk_inference_engine.py` already does, was statistically indistinguishable from CC-only,
and degrades gracefully when a view is missing.

Every number written below is read back out of `fusion_summary.csv` and
`fusion_subgroups.csv`. Nothing is retyped from a printout.

**No inference runs here.** The sealed holdout is not opened and no threshold is calibrated.

In [54]:
import os, json, numpy as np, pandas as pd

if 'OUT_DIR' not in globals():
    OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'


def _find(fname):
    p = os.path.join(OUT_DIR, fname)
    if os.path.exists(p):
        return p
    for root in ('/kaggle/input', '.'):
        if os.path.isdir(root):
            for dp, _d, fs in os.walk(root):
                if fname in fs:
                    return os.path.join(dp, fname)
    return None


_need = ['fusion_summary.csv', 'fusion_subgroups.csv', 'frozen_fusion.json']
_paths = {f: _find(f) for f in _need}
_miss = [f for f, p in _paths.items() if p is None]
if _miss:
    raise SystemExit('Missing fusion outputs: %s. Run the fusion cells first, or attach '
                     'the saved files.' % ', '.join(_miss))

SUM = pd.read_csv(_paths['fusion_summary.csv'])
SUB = pd.read_csv(_paths['fusion_subgroups.csv'])
OLD = json.load(open(_paths['frozen_fusion.json']))

ENS = 'ensemble' if (SUM.scorer == 'ensemble').any() else SUM.scorer.iloc[-1]
E = SUM[SUM.scorer == ENS].set_index('method')
SCORERS = list(dict.fromkeys(SUM.scorer))
print('loaded fusion outputs from:', os.path.dirname(_paths['fusion_summary.csv']))
print('scorers:', SCORERS, '| ensemble column:', ENS)
print('breasts %d, cancer %d, non-cancer %d'
      % (E.loc['cc_only', 'n_breasts'], E.loc['cc_only', 'pos'], E.loc['cc_only', 'neg']))

loaded fusion outputs from: /kaggle/working
scorers: ['v11_dro', 'v8_resnet', 'ensemble'] | ensemble column: ensemble
breasts 1705, cancer 343, non-cancer 1362


## P1. Re-derive the supporting numbers from the saved tables

Four claims go into the report. Each is computed here rather than copied, so the report
cannot drift from the data behind it.

In [55]:
# --- 1. no method beat the stronger single view with an interval excluding zero ---
strong = E.loc['mean_prob', 'stronger_single']
FUSIONS = ['mean_prob', 'max_prob', 'mean_logit']
supported = [m for m in FUSIONS
             if E.loc[m, 'auc'] > E.loc['cc_only', 'auc']
             and E.loc[m, 'auc'] > E.loc['mlo_only', 'auc']
             and E.loc[m, 'vs_stronger_lo'] > 0]
print('1. stronger single view: %s (AUC %.4f)' % (strong, E.loc[strong, 'auc']))
for m in FUSIONS:
    print('   %-11s AUC %.4f   vs %s %+.4f [%+.4f, %+.4f]'
          % (m, E.loc[m, 'auc'], strong, E.loc[m, 'vs_stronger'],
             E.loc[m, 'vs_stronger_lo'], E.loc[m, 'vs_stronger_hi']))
print('   methods passing the pre-registered rule:', supported or 'none')

# --- 2. max_prob against cc_only, the operational question ---
mx = E.loc['max_prob']
indistinguishable = bool(mx['vs_cc_only_lo'] < 0 < mx['vs_cc_only_hi'])
print()
print('2. max_prob vs cc_only: %+.4f [%+.4f, %+.4f]  -> %s'
      % (mx['vs_cc_only'], mx['vs_cc_only_lo'], mx['vs_cc_only_hi'],
         'statistically indistinguishable' if indistinguishable else 'DIFFERENT - re-read'))

# --- 3. every subgroup a fusion method significantly harms (CI entirely below zero) ---
sig = SUB[(SUB.scorer == ENS) & (SUB.method.isin(FUSIONS)) &
          (SUB.pos >= 10) & (SUB.neg >= 10) & (SUB.hi < 0)]
print()
print('3. subgroups significantly harmed by a fusion method (ensemble, CI entirely < 0):')
if len(sig):
    print(sig[['stratum', 'method', 'n_breasts', 'pos', 'neg', 'vs_stronger', 'lo', 'hi']]
          .round(4).to_string(index=False))
else:
    print('   none')

# --- 4. strata too small to evaluate ---
tiny = SUB[SUB.note.astype(str).str.contains('too few', na=False)][
    ['stratum', 'n_breasts', 'pos', 'neg']].drop_duplicates('stratum')
print()
print('4. strata that could not be evaluated:')
print(tiny.to_string(index=False))

# --- 5. CC minus MLO in every scorer, as an observation only ---
ccmlo = {s: float(SUM[(SUM.scorer == s) & (SUM.method == 'cc_only')].iloc[0]['vs_mlo_only'])
         for s in SCORERS}
print()
print('5. cc_only minus mlo_only, by scorer:',
      '  '.join('%s %+.4f' % (k, v) for k, v in ccmlo.items()))

1. stronger single view: cc_only (AUC 0.6030)
   mean_prob   AUC 0.5952   vs cc_only -0.0078 [-0.0235, +0.0087]
   max_prob    AUC 0.6050   vs cc_only +0.0020 [-0.0111, +0.0158]
   mean_logit  AUC 0.5975   vs cc_only -0.0055 [-0.0204, +0.0103]
   methods passing the pre-registered rule: none

2. max_prob vs cc_only: +0.0020 [-0.0111, +0.0158]  -> statistically indistinguishable

3. subgroups significantly harmed by a fusion method (ensemble, CI entirely < 0):
     stratum     method  n_breasts  pos  neg  vs_stronger      lo      hi
age_band <50  mean_prob        331   30  301      -0.0658 -0.1257 -0.0118
age_band <50 mean_logit        331   30  301      -0.0614 -0.1169 -0.0096

4. strata that could not be evaluated:
    stratum  n_breasts  pos  neg
  density D         50    4   46
 machine 93         56    4   52
machine 170         27    7   20
machine 190          9    1    8
machine 197          2    0    2
machine 210         38    0   38
machine 216         50    1   49

5. cc_onl

## P2. Rewrite `frozen_fusion.json`

The operational rule is stated so it can be implemented without reading this notebook, and
the reference implementation below is exercised on the four cases that matter: both views,
CC only, MLO only, neither.

In [56]:
def fuse_breast(cc_prob=None, mlo_prob=None):
    """Frozen breast-level rule: maximum over whichever views exist.

    cc_prob / mlo_prob are the within-view mean probabilities for ONE breast
    (one patient, one laterality). Left and right are never combined.
    Returns None when neither view is present.
    """
    vals = [p for p in (cc_prob, mlo_prob) if p is not None and not pd.isna(p)]
    return max(vals) if vals else None


assert fuse_breast(0.3, 0.7) == 0.7
assert fuse_breast(0.8, 0.2) == 0.8
assert fuse_breast(0.4, None) == 0.4
assert fuse_breast(None, 0.6) == 0.6
assert fuse_breast(None, None) is None
assert fuse_breast(0.5, float('nan')) == 0.5
print('fuse_breast self-test: ALL PASSED')

frozen = {
    'method': 'max_available_view_probability',
    'status': 'retained operational rule, not a demonstrated AUC improvement',
    'breast_key': 'patient_id + "_" + laterality',
    'rule': {
        'both_views_present': 'use max(CC, MLO)',
        'single_view_present': 'use the available view probability',
        'no_view_present': 'no breast-level score is produced',
        'left_and_right': 'never combined - separate breasts, separate scores',
        'within_view_duplicates': 'mean of the predictions of that view, applied before fusion',
    },
    'rationale': (
        'No fusion method significantly outperformed CC-only: none beat both single views '
        'with a paired 95%% interval excluding zero. max_prob was statistically '
        'indistinguishable from CC-only (%+.4f [%+.4f, %+.4f]) and is the rule already '
        'implemented in risk_inference_engine.py. CC-only is not deployable because breasts '
        'holding only MLO cannot be scored by it.'
        % (mx['vs_cc_only'], mx['vs_cc_only_lo'], mx['vs_cc_only_hi'])),
    'fusion_supported': False,
    'stronger_single_view': strong,
    'ensemble_auc_breast_level': {m: float(E.loc[m, 'auc']) for m in E.index},
    'max_prob_vs_cc_only': {'delta': float(mx['vs_cc_only']),
                            'lo': float(mx['vs_cc_only_lo']),
                            'hi': float(mx['vs_cc_only_hi']),
                            'interpretation': 'no measurable difference'},
    'cc_minus_mlo_by_scorer': ccmlo,
    'cohort': {'n_breasts_both_views': int(E.loc['cc_only', 'n_breasts']),
               'n_cancer_breasts': int(E.loc['cc_only', 'pos']),
               'n_noncancer_breasts': int(E.loc['cc_only', 'neg'])},
    'subgroups_significantly_harmed_by_fusion': (
        sig[['stratum', 'method', 'vs_stronger', 'lo', 'hi']].to_dict('records')),
    'subgroups_not_evaluable': tiny.to_dict('records'),
    'evaluated_on': 'RSNA development split only; holdout sealed and not opened',
    'windowing': OLD.get('windowing', 'wide 0.1-99.9, frozen'),
    'bootstrap': OLD.get('bootstrap'),
    'logistic_fusion': OLD.get('logistic_fusion'),
    'threshold_calibration': 'NOT performed in this patch - next task',
    'supersedes': {'previous_decision': OLD.get('decision'),
                   'reason': 'cc_only is a finding, not an implementable rule'},
}

with open(os.path.join(OUT_DIR, 'frozen_fusion.json'), 'w') as f:
    json.dump(frozen, f, indent=2, default=float)
print('rewrote frozen_fusion.json')
print(json.dumps({k: frozen[k] for k in ('method', 'status', 'breast_key', 'rule',
                                         'fusion_supported')}, indent=2))

fuse_breast self-test: ALL PASSED
rewrote frozen_fusion.json
{
  "method": "max_available_view_probability",
  "status": "retained operational rule, not a demonstrated AUC improvement",
  "breast_key": "patient_id + \"_\" + laterality",
  "rule": {
    "both_views_present": "use max(CC, MLO)",
    "single_view_present": "use the available view probability",
    "no_view_present": "no breast-level score is produced",
    "left_and_right": "never combined - separate breasts, separate scores",
    "within_view_duplicates": "mean of the predictions of that view, applied before fusion"
  },
  "fusion_supported": false
}


## P3. Rewrite the decision report

Four statements, each carrying the number that supports it.

In [57]:
L = []
A = L.append
A('# Sprint 5 - breast-level CC/MLO fusion: decision')
A('')
A('**Finding: fusion was not supported.**  ')
A('**Operational rule: `max_available_view_probability` - retained, not an improvement.**')
A('')
A('## Cohort')
A('')
A('- %d breasts holding both CC and MLO, %d cancer breasts, %d non-cancer.'
  % (E.loc['cc_only', 'n_breasts'], E.loc['cc_only', 'pos'], E.loc['cc_only', 'neg']))
A('- RSNA development split only. The sealed holdout was not opened.')
A('- Predictions come from the frozen wide 0.1-99.9 windowing pass. No image was re-scored.')
A('- Duplicate views within a breast were averaged before fusion, so the single-view '
  'baselines contain no hidden second fusion step.')
A('')
A('## Ensemble, breast level')
A('')
A('| method | AUC | 95% CI | vs ' + strong + ' | 95% CI |')
A('|---|---|---|---|---|')
for m in ['cc_only', 'mlo_only', 'mean_prob', 'max_prob', 'mean_logit']:
    A('| `%s` | %.4f | %.4f to %.4f | %+.4f | %+.4f to %+.4f |'
      % (m, E.loc[m, 'auc'], E.loc[m, 'auc_lo'], E.loc[m, 'auc_hi'],
         E.loc[m, 'vs_stronger'], E.loc[m, 'vs_stronger_lo'], E.loc[m, 'vs_stronger_hi']))
A('')
A('## 1. Fusion was not supported')
A('')
A('No fusion method beat both single views with a paired 95%% interval excluding zero. '
  'The strongest single view was `%s` at %.4f. The closest fusion method, `max_prob`, '
  'reached %.4f, a difference of %+.4f [%+.4f, %+.4f] against it.'
  % (strong, E.loc[strong, 'auc'], E.loc['max_prob', 'auc'],
     E.loc['max_prob', 'vs_stronger'], E.loc['max_prob', 'vs_stronger_lo'],
     E.loc['max_prob', 'vs_stronger_hi']))
A('')
A('This answers the client question about whether the system needs one image or several: '
  'on external screening data, combining the two standard views did not improve ranking.')
A('')
A('The operational rule is nevertheless the maximum over available views, because `cc_only` '
  'cannot score a breast that has no CC image, `max_prob` was statistically '
  'indistinguishable from it (%+.4f [%+.4f, %+.4f]), and it is already what the inference '
  'engine implements. That is an engineering decision made on the absence of a measurable '
  'difference, and it is deliberately separate from the pre-registered adoption rule above.'
  % (mx['vs_cc_only'], mx['vs_cc_only_lo'], mx['vs_cc_only_hi']))
A('')
A('## 2. Mean probability harmed the under-50 subgroup')
A('')
if len(sig):
    A('| stratum | method | breasts | cancer | non-cancer | delta vs stronger | 95% CI |')
    A('|---|---|---|---|---|---|---|')
    for _, r in sig.iterrows():
        A('| %s | `%s` | %d | %d | %d | %+.4f | %+.4f to %+.4f |'
          % (r['stratum'], r['method'], r['n_breasts'], r['pos'], r['neg'],
             r['vs_stronger'], r['lo'], r['hi']))
    A('')
    A('These intervals lie entirely below zero, so the harm is not attributable to noise. '
      'It did not trip the pre-registered harm criterion only because that criterion asked '
      'for an upper bound below -0.02 and these sit just above it. The threshold was '
      'therefore looser than the evidence warranted, and this is recorded as a weakness in '
      'the rule rather than corrected after the fact.')
else:
    A('No subgroup was significantly harmed by a fusion method at the sizes available.')
A('')
A('## 3. Density D could not be evaluated')
A('')
_dD = tiny[tiny.stratum.astype(str).str.contains('density D', case=False, na=False)]
if len(_dD):
    r = _dD.iloc[0]
    A('Density D held %d breasts with only %d cancer breasts, below the minimum of ten '
      'positives and ten negatives required for a stable estimate. No AUC is reported for '
      'it.' % (r['n_breasts'], r['pos']))
else:
    A('Density D fell below the minimum class counts required for a stable estimate.')
A('')
A('This matters beyond fusion: density D is one of the two subgroups scoring below chance '
  'in external validation, and it has not been evaluable at any point in this sprint. It '
  'belongs in the model card as a stated limitation, not as a number.')
A('')
A('## 4. The CC-MLO difference is an observation only')
A('')
A('CC scored above MLO in every scorer: %s. MLO is the view containing the pectoral muscle '
  'that this pipeline does not remove, so the direction is consistent with the pectoral '
  'hypothesis parked earlier in the sprint.'
  % ', '.join('%s %+.4f' % (k, v) for k, v in ccmlo.items()))
A('')
A('It is not evidence for it. The comparison was not designed to test that hypothesis, no '
  'interval excludes zero, and view difficulty differs for reasons unrelated to the muscle. '
  'It is recorded as an observation worth a designed test later, and nothing more.')
A('')
A('## Scope')
A('')
A('- The sealed RSNA holdout was not opened.')
A('- No threshold was calibrated. That is the next task.')
A('- Logistic-regression fusion was not fitted: its weights require a Mammo-Bench internal '
  'validation set, and fitting them on RSNA development data would place the fusion rule '
  'and its evaluation on the same dataset.')
A('')

_rep = os.path.join(OUT_DIR, 'fusion_decision.md')
with open(_rep, 'w') as f:
    f.write('\n'.join(L))
print('rewrote fusion_decision.md (%d lines)' % len(L))
print()
print('\n'.join(L))

rewrote fusion_decision.md (57 lines)

# Sprint 5 - breast-level CC/MLO fusion: decision

**Finding: fusion was not supported.**  
**Operational rule: `max_available_view_probability` - retained, not an improvement.**

## Cohort

- 1705 breasts holding both CC and MLO, 343 cancer breasts, 1362 non-cancer.
- RSNA development split only. The sealed holdout was not opened.
- Predictions come from the frozen wide 0.1-99.9 windowing pass. No image was re-scored.
- Duplicate views within a breast were averaged before fusion, so the single-view baselines contain no hidden second fusion step.

## Ensemble, breast level

| method | AUC | 95% CI | vs cc_only | 95% CI |
|---|---|---|---|---|
| `cc_only` | 0.6030 | 0.5675 to 0.6373 | +0.0000 | +0.0000 to +0.0000 |
| `mlo_only` | 0.5749 | 0.5407 to 0.6099 | -0.0281 | -0.0579 to +0.0029 |
| `mean_prob` | 0.5952 | 0.5609 to 0.6289 | -0.0078 | -0.0235 to +0.0087 |
| `max_prob` | 0.6050 | 0.5700 to 0.6397 | +0.0020 | -0.0111 to +0.0158 |
| `mean_logit`

## P4. Week 2 task closed

`frozen_fusion.json` and `fusion_decision.md` are rewritten. Download both again - they
replace the versions produced by the fusion run.

The fusion rule is frozen. Threshold calibration is next, and it starts from these frozen
breast-level scores, on the development set, with the holdout still sealed.